# Clase 124 — tf.data API

Construimos **pipelines de datos eficientes** con `tf.data.Dataset`: leer,
`map`, `shuffle`, `batch` y `prefetch`. Un buen pipeline es la diferencia entre
"GPU al 30 %" y "GPU al 95 %".

Requiere: `tensorflow` / `keras` (≥ 3.0).

## 1. Dataset desde memoria: `from_tensor_slices`

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
keras.utils.set_random_seed(42)
AUTOTUNE = tf.data.AUTOTUNE

(X_tr, y_tr), _ = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.astype("float32")

ds = tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
print("element_spec:", ds.element_spec)
for imagen, etiqueta in ds.take(1):
    print("primera imagen:", imagen.shape, "| etiqueta:", int(etiqueta))

## 2. `map`: transformar cada elemento

In [ ]:
def normalizar(imagen, etiqueta):
    imagen = tf.cast(imagen, tf.float32) / 255.0
    imagen = tf.reshape(imagen, (784,))
    return imagen, etiqueta

ds_norm = ds.map(normalizar, num_parallel_calls=AUTOTUNE)
x, y = next(iter(ds_norm))
print("tras map:", x.shape,
      "| rango:", float(tf.reduce_min(x)), "-", float(tf.reduce_max(x)))

## 3. `shuffle`, `batch`, `prefetch`

In [ ]:
pipeline = (ds_norm
            .shuffle(buffer_size=1024)
            .batch(128)
            .prefetch(AUTOTUNE))

xb, yb = next(iter(pipeline))
print("batch X:", xb.shape, "| batch y:", yb.shape)
print("batches totales:", len(pipeline))

## 4. `cache`, `repeat` y `AUTOTUNE` (orden canónico)

In [ ]:
# orden canónico: map -> cache -> shuffle -> batch -> prefetch
pipeline_cache = (ds.map(normalizar, num_parallel_calls=AUTOTUNE)
                  .cache()                 # cachea tras el map (no re-normaliza)
                  .shuffle(1024)
                  .batch(128)
                  .prefetch(AUTOTUNE))
# repeat para entrenar con steps_per_epoch fijos:
pipeline_repeat = pipeline_cache.repeat(2)
print("cache + repeat listo | AUTOTUNE =", AUTOTUNE)

## 5. `filter` e `interleave` (leer varios archivos en paralelo)

In [ ]:
# filter: quedarse solo con las clases 0..4
ds_filtrado = ds_norm.filter(lambda x, y: y < 5)
print("primeras etiquetas filtradas (<5):",
      [int(y) for _, y in ds_filtrado.take(5)])

# interleave: leer varias fuentes en paralelo (aquí simulado)
archivos = tf.data.Dataset.from_tensor_slices(["a", "b", "c"])
entrelazado = archivos.interleave(
    lambda f: tf.data.Dataset.from_tensor_slices([f + "-1", f + "-2"]),
    cycle_length=3, num_parallel_calls=AUTOTUNE)
print("interleave:", [s.decode() for s in entrelazado.as_numpy_iterator()])

## 6. Pipeline production-ready + `fit`

In [ ]:
def aumentar(imagen, etiqueta):
    imagen = tf.image.random_flip_left_right(tf.reshape(imagen, (28, 28, 1)))
    return tf.reshape(imagen, (784,)), etiqueta

train_ds = (ds.map(normalizar, num_parallel_calls=AUTOTUNE)
            .cache()
            .map(aumentar, num_parallel_calls=AUTOTUNE)
            .shuffle(2048)
            .batch(128)
            .prefetch(AUTOTUNE))

modelo = keras.Sequential([
    keras.Input((784,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
modelo.compile(optimizer="adam",
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.fit(train_ds, epochs=5, verbose=2)

## Ejercicios

1. **Dataset desde NumPy**: armá `from_tensor_slices((x, y)).shuffle(1024)
   .batch(32).prefetch(AUTOTUNE)` e iterá verificando shapes.
2. **Map con normalización**: aplicá `.map(lambda x, y: (tf.cast(x, tf.float32)/255., y),
   num_parallel_calls=AUTOTUNE)`.
3. **Cache**: comparación de tiempo del 1er epoch vs el 2do con y sin `.cache()`.
4. **Buffer chico**: comparación de `shuffle(10)` vs `shuffle(len(data))`
   inspeccionando el primer batch.
5. **Orden canónico**: verificá por qué `.cache().shuffle()` es correcto y
   `.shuffle().cache()` congela el orden.

## Conclusiones

- `tf.data.Dataset` es **lazy**: describe un grafo de transformaciones, no datos ya cargados.
- Orden canónico: `map -> cache -> shuffle -> batch -> prefetch`.
- `num_parallel_calls=AUTOTUNE` y `prefetch(AUTOTUNE)` solapan CPU (loading) con GPU (compute).
- `cache()` vale la pena cuando el dataset entra en RAM: el 2do epoch salta I/O y `map`.
- `shuffle(buffer)` con buffer chico mezcla mal; usar ≥ 10 % del dataset (o el tamaño completo).

## ✅ Soluciones de los ejercicios

`tf.data` API (cap. 13). Se valida por AST sin TF; los ejercicios de `cache`/`shuffle` usan datasets sintéticos (`tf.data.Dataset.range`) para que sean autocontenidos. Cubren pipeline básico, `map` paralelo, efecto de `cache`, tamaño del buffer de shuffle y profiling.

**Ej. 1 — Dataset desde NumPy.** `from_tensor_slices -> shuffle -> batch -> prefetch`.

In [ ]:
import tensorflow as tf
from tensorflow import keras

(x_train, y_train), _ = keras.datasets.fashion_mnist.load_data()
ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
      .shuffle(1024).batch(32).prefetch(tf.data.AUTOTUNE))
for xb, yb in ds.take(1):
    print(xb.shape, yb.shape)   # (32, 28, 28) (32,)

**Ej. 2 — Map con normalización.** `map(..., num_parallel_calls=AUTOTUNE)` para paralelizar.

In [ ]:
import tensorflow as tf

ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255., y),
            num_parallel_calls=tf.data.AUTOTUNE)
for xb, _ in ds.take(1):
    print("rango:", float(tf.reduce_min(xb)), "..", float(tf.reduce_max(xb)))   # ~0 .. 1

**Ej. 3 — Cache.** Comparar 1a vs 2a época con y sin `.cache()` (dataset sintético autocontenido).

In [ ]:
import time
import tensorflow as tf

base = tf.data.Dataset.range(10_000).map(lambda x: x * 2)   # map "caro"
cached = base.cache()

def epoch_time(d):
    t0 = time.perf_counter()
    for _ in d:
        pass
    return time.perf_counter() - t0

print(f"sin cache: 1a {epoch_time(base):.3f}s | 2a {epoch_time(base):.3f}s")
print(f"con cache: 1a {epoch_time(cached):.3f}s | 2a {epoch_time(cached):.3f}s  <- 2a mas rapida")

**Ej. 4 — Buffer chico vs grande.** `shuffle(10)` mezcla poco; `shuffle(N)` mezcla de verdad.

In [ ]:
import tensorflow as tf

data = tf.data.Dataset.range(100)
small = list(data.shuffle(10).batch(10).take(1))[0].numpy()    # buffer=10
full = list(data.shuffle(100).batch(10).take(1))[0].numpy()    # buffer=100
print("buffer=10 :", small)   # valores bajos agrupados (poco aleatorio)
print("buffer=100:", full)    # realmente barajado
# regla: buffer_size >= tamaño del dataset para un shuffle uniforme

**Ej. 5 — Profilear.** Callback de TensorBoard con `profile_batch`; si domina 'Input', el cuello es I/O.

In [ ]:
import tensorflow as tf

tb = tf.keras.callbacks.TensorBoard(log_dir="logs", profile_batch=(10, 20))
# model.fit(ds, callbacks=[tb]);  luego: tensorboard --logdir logs  -> pestaña Profile.
# Si el trace muestra el device esperando datos (Input-bound) -> agregar
# prefetch(AUTOTUNE), cache() y map(num_parallel_calls=AUTOTUNE).
print("profile_batch=(10,20): traza los batches 10-20 y revela si el cuello es datos o compute")